In [1]:
import pandas as pd
import numpy as np

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import joblib
import os

In [2]:
X_train = pd.read_csv("../data/normalized/X_train_StandardScaler.csv")

X_test = pd.read_csv("../data/normalized/X_test_StandardScaler.csv")

y_train = pd.read_csv("../data/imbalanced/y_SMOTE.csv").values.ravel()

y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (1768, 11)
Test Shape: (333, 11)


In [3]:
models = {

    "RandomForest":
        RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            eval_metric="logloss",
            use_label_encoder=False,
            random_state=42
        ),

    "SVM":
        SVC(),

    "LogisticRegression":
        LogisticRegression(
            max_iter=1000
        ),

    "NaiveBayes":
        GaussianNB()
}

In [4]:
results = []

trained_models = {}

for model_name, model in models.items():

    print(f"\n🚀 Training: {model_name}")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)

    prec = precision_score(
        y_test,
        y_pred,
        average="macro"
    )

    rec = recall_score(
        y_test,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro"
    )

    results.append({

        "Model": model_name,

        "Accuracy": round(acc, 4),

        "Precision": round(prec, 4),

        "Recall": round(rec, 4),

        "F1-Score": round(f1, 4)
    })

    trained_models[model_name] = model

    print("✅ Done")


🚀 Training: RandomForest
✅ Done

🚀 Training: XGBoost
✅ Done

🚀 Training: SVM


c:\Users\teler\miniconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [14:38:44] WARNING: C:\Users\task_177740979073858\croot\xgboost-split_1777409985184\work\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Done

🚀 Training: LogisticRegression
✅ Done

🚀 Training: NaiveBayes
✅ Done


In [5]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1-Score",
    ascending=False
)

print("\n📊 FINAL MODEL COMPARISON RESULTS\n")

print(results_df)


📊 FINAL MODEL COMPARISON RESULTS

                Model  Accuracy  Precision  Recall  F1-Score
0        RandomForest       1.0        1.0     1.0       1.0
1             XGBoost       1.0        1.0     1.0       1.0
2                 SVM       1.0        1.0     1.0       1.0
3  LogisticRegression       1.0        1.0     1.0       1.0
4          NaiveBayes       1.0        1.0     1.0       1.0


In [6]:
os.makedirs("../reports", exist_ok=True)

results_df.to_csv(
    "../reports/final_model_results.csv",
    index=False
)

print("✅ Results Saved")

✅ Results Saved


In [7]:
best_model_name = results_df.iloc[0]["Model"]

print("Best Model:", best_model_name)

best_model = trained_models[best_model_name]

Best Model: RandomForest


In [8]:
y_pred = best_model.predict(X_test)

print("\n📊 CLASSIFICATION REPORT\n")

print(
    classification_report(
        y_test,
        y_pred
    )
)


📊 CLASSIFICATION REPORT

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       222
           1       1.00      1.00      1.00       111

    accuracy                           1.00       333
   macro avg       1.00      1.00      1.00       333
weighted avg       1.00      1.00      1.00       333



In [9]:
cm = confusion_matrix(y_test, y_pred)

print("\n📊 CONFUSION MATRIX\n")

print(cm)


📊 CONFUSION MATRIX

[[222   0]
 [  0 111]]


In [10]:
os.makedirs("../model", exist_ok=True)

joblib.dump(
    best_model,
    "../model/final_best_model.pkl"
)

print("✅ Best Model Saved")

✅ Best Model Saved


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
import pandas as pd
import os
import joblib

# Define feature columns
categorical_cols = ["ip_type", "source"]
numerical_cols = [
    "inter_api_access_duration(sec)",
    "api_access_uniqueness",
    "sequence_length(count)",
    "vsession_duration(min)",
    "num_sessions",
    "num_users",
    "num_unique_apis",
]

# Load raw training data
X_train_raw = pd.read_csv("../data/processed/X_train.csv")
y_train_raw = pd.read_csv("../data/processed/y_train.csv").values.ravel()

print("Raw training data shape:", X_train_raw.shape)
print("Raw training labels shape:", y_train_raw.shape)

# Create preprocessor with imputation, scaling, and encoding
preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]),
        numerical_cols,
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]),
        categorical_cols,
    ),
])

# Create full deployment pipeline
deployment_pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42)),
])

# Train on raw data
print("\n🚀 Training deployment pipeline on raw data...")
deployment_pipeline.fit(X_train_raw, y_train_raw)
print("✅ Pipeline trained successfully")

# Save the full pipeline
os.makedirs("../model", exist_ok=True)
joblib.dump(deployment_pipeline, "../model/api_security_pipeline.pkl")
print("✅ Full deployment pipeline saved to ../model/api_security_pipeline.pkl")

# Verify it loads correctly
test_pipeline = joblib.load("../model/api_security_pipeline.pkl")
print("✅ Pipeline loaded and verified successfully")

Raw training data shape: (1329, 9)
Raw training labels shape: (1329,)

🚀 Training deployment pipeline on raw data...
✅ Pipeline trained successfully
✅ Full deployment pipeline saved to ../model/api_security_pipeline.pkl
✅ Pipeline loaded and verified successfully
